# Pièges pandas

> Notebook annexe — à lire quand un comportement vous surprend ou quand un bug résiste.
> Tous ces pièges sont réels et ont causé des erreurs en production.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from _data import load_penguins

penguins = load_penguins()

## 1. Vue vs copie et Copy-on-Write

C'était historiquement le piège numéro un en pandas : modifier une "vue" sans affecter
le DataFrame original, ou l'inverse.

**Depuis pandas 2.0, Copy-on-Write (CoW) est activé par défaut.**
CoW garantit que toute modification crée une copie implicite — les anciens bugs de view silencieux disparaissent.
Mais il faut comprendre le mécanisme pour ne pas être surpris.

In [ ]:
print("Copy-on-Write activé :", pd.options.mode.copy_on_write)

In [ ]:
# Avant CoW (pandas < 2.0) : ce code pouvait modifier df ou non, selon l'humeur interne
# Avec CoW : subset est toujours une copie indépendante dès qu'on le modifie

df_orig = pd.DataFrame({"a": [1, 2, 3], "b": [10, 20, 30]})
subset = df_orig[["a", "b"]].query("a > 1")

subset.iloc[0, 0] = 999  # modifie subset

print("df_orig intact :", df_orig["a"].tolist())
print("subset modifié :", subset["a"].tolist())

In [ ]:
# Conséquence de CoW : pour modifier le DataFrame original,
# il faut assigner explicitement avec .loc[]

df_orig.loc[df_orig["a"] > 1, "b"] = 999
print("df_orig après .loc[] :", df_orig["b"].tolist())

## 2. L'index qui traîne

Après un filtre ou un tri, l'index garde ses valeurs originales.
C'est source de bugs quand on utilise `.iloc[]` en croyant utiliser `.loc[]`
ou quand on `concat` deux DataFrames sans `ignore_index`.

In [ ]:
# Avant d'exécuter — que retourne .iloc[0] ?
df_filtre = penguins.query("species == 'Gentoo'")
print("Premier label d'index :", df_filtre.index[0])
print(".iloc[0] :", df_filtre.iloc[0]["species"])  # bonne ligne du sous-ensemble
print(".loc[0]  :", end=" ")
try:
    print(df_filtre.loc[0]["species"])  # cherche le label 0 — qui n'existe pas
except KeyError as e:
    print(f"KeyError {e} — label 0 absent après filtre")

In [ ]:
# Concat sans ignore_index : doublons d'index
part1 = penguins.head(3)
part2 = penguins.head(3)  # mêmes lignes = même index
concat_naif = pd.concat([part1, part2])
print("Index :", concat_naif.index.tolist())
print("Doublons :", concat_naif.index.duplicated().sum())

In [ ]:
# Solution : ignore_index=True ou .reset_index(drop=True)
pd.concat([part1, part2], ignore_index=True).index.tolist()

## 3. Les approximations flottantes

Les flottants ne peuvent pas représenter exactement toutes les valeurs décimales.
C'est une limitation de l'arithmétique binaire, pas de pandas.

In [ ]:
# Avant d'exécuter — que retourne 0.1 + 0.2 == 0.3 ?
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)  # False !

# Conséquence : comparer des flottants avec == est risqué
s = pd.Series([0.1 + 0.2, 0.3, 0.6])
print(s == 0.3)

In [ ]:
# Solution : utiliser np.isclose() ou .round() avant comparaison
print(np.isclose(s, 0.3))
print(s.round(10) == 0.3)

## 4. `groupby` ignore les NaN en silence

Déjà vu dans le notebook 4.0, mais suffisamment trompeur pour le répéter ici.

In [ ]:
print(f"Total pingouins         : {len(penguins)}")
print(f"NaN dans sex            : {penguins['sex'].isna().sum()}")
print(f"Comptés par groupby(sex): {penguins.groupby('sex')['species'].count().sum()}")
print()
print("La différence :", len(penguins) - penguins.groupby("sex")["species"].count().sum(),
      "pingouins oubliés silencieusement")

In [ ]:
# Correction
penguins.groupby("sex", dropna=False)["species"].count()

## 5. `.apply()` retourne des types inattendus

In [ ]:
# apply() essaie d'inférer le type de retour — parfois surprenant
df_test = pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})

# Retourne une Series
result = df_test.apply(lambda df_: df_.sum(), axis=0)
print(type(result), result.tolist())

In [ ]:
# Si la fonction retourne des Series de longueurs différentes → comportement variable
# Préférez result_type='expand' pour être explicite
def retourner_serie(row):
    return pd.Series({"somme": row.sum(), "max": row.max()})

df_test.apply(retourner_serie, axis=1)

## 6. Mutation d'un DataFrame passé en argument

En Python, les DataFrames sont passés par référence.
Une fonction qui modifie son argument peut modifier l'objet original sans prévenir.

In [ ]:
def ajouter_colonne(d):
    d["nouvelle"] = 42  # modifie l'original avec CoW=False
    return d

original = pd.DataFrame({"a": [1, 2, 3]})
_ = ajouter_colonne(original)

# Avec CoW=True (pandas 2.0+), l'original est protégé
# Sans CoW, l'original serait modifié
print("Colonnes de l'original :", original.columns.tolist())

In [ ]:
# Bonne pratique : utiliser .assign() qui retourne toujours un nouvel objet
def ajouter_colonne_safe(d):
    return d.assign(nouvelle=42)  # ne modifie jamais l'argument

original2 = pd.DataFrame({"a": [1, 2, 3]})
result = ajouter_colonne_safe(original2)
print("Original intouché :", original2.columns.tolist())
print("Résultat          :", result.columns.tolist())

## 7. Chaînes de caractères vs entiers dans les comparaisons

Quand pandas lit un CSV, il infère les types. Une colonne de codes départements
peut être lue comme `int64` ou comme `object` selon la présence de valeurs
comme `"2A"` ou `"2B"` (Corse).

In [ ]:
# Simulation du piège : même colonne, types différents
s_int = pd.Series([75, 69, 13], dtype=int)
s_str = pd.Series(["75", "69", "13"], dtype=str)

print("Comparaison int vs int :", (s_int == 75).sum())
print("Comparaison str vs int :", (s_str == 75).sum())   # 0 ! '75' != 75
print("Comparaison str vs str :", (s_str == "75").sum()) # 1

In [ ]:
# Diagnostic rapide : vérifier le dtype avant de filtrer
from _data import load_accidents
df = load_accidents()
print("dtype de dep :", df["dep"].dtype)
print("Valeur exemple :", repr(df["dep"].iloc[0]))

## 8. `inplace=True` — pourquoi on ne l'utilise plus

In [ ]:
# inplace=True modifie l'objet ET retourne None
# → impossible à chaîner
# → deprecated en pandas 2.0 sur certaines méthodes

df_test = penguins.copy()
result = df_test.dropna(inplace=True)  # modifie df_test, retourne None
print("result :", result)  # None — pas chaînable

# Bonne pratique
df_propre = penguins.dropna()  # retourne un nouveau DataFrame
print("df_propre shape :", df_propre.shape)

## 9. `.value_counts()` n'inclut pas les NaN par défaut

In [ ]:
# Silencieux — les NaN sont exclus du compte par défaut
print("Avec dropna=True  (défaut) :", penguins["sex"].value_counts().sum())
print("Avec dropna=False          :", penguins["sex"].value_counts(dropna=False).sum())
print("Total                      :", len(penguins))

## Récapitulatif — pièges les plus fréquents

| Piège | Symptôme | Correction |
|---|---|---|
| Modifier une view | Modification sans effet ou effet inattendu | Utiliser `.loc[]`, `.assign()` ou `.copy()` |
| Index qui traîne après filtre | `.loc[0]` → KeyError, `concat` → doublons | `.reset_index(drop=True)`, `ignore_index=True` |
| Comparaison flottante avec `==` | `0.1 + 0.2 == 0.3` → False | `np.isclose()` ou `.round()` |
| `groupby` ignore les NaN | Comptages faux | `groupby(dropna=False)` |
| `value_counts()` ignore les NaN | Proportions fausses | `value_counts(dropna=False)` |
| String vs int dans filtres | `dep == 75` trouve 0 lignes si dep est string | Vérifier `df["col"].dtype` avant de filtrer |
| `inplace=True` non chaînable | Retourne `None` | Réassigner : `df = df.dropna()` |
| Fonction qui mute l'argument | L'original est modifié | Utiliser `.assign()` dans la fonction |